# Referencia histórica del Taller 3
Autoría: Diego Linares, Andy Fuentes, Diederich Solís y Christian Echeverria. Fuente: https://github.com/DiegoLinares11/MLFlow, commit bd619c3cea3eb2d00fb6732e2993530d87de3f39. Las salidas conservadas proceden del taller original; no son resultados de esta adaptación ni de Databricks. Para la entrega ejecute Champions_Databricks.ipynb.

# Taller 3 · MLflow con un pipeline de scikit-learn

Diego Linares · Andy Fuentes · Christian Echeverria · Diederich Solis

Conectamos MLflow con el pipeline de scikit-learn que venimos usando desde la Actividad 1 (dataset de
la UEFA Champions League del Ejercicio 1, versión de la Actividad 9). El objetivo es predecir el
resultado de un partido (gana el local, gana el visitante o empate) a partir de sus estadísticas.

## Qué dice la documentación

La sección que explica cómo usar las dos herramientas juntas es la **integración de MLflow con
scikit-learn**:

- Página general: <https://mlflow.org/docs/latest/ml/traditional-ml/sklearn/>
- Guía completa, con la sección **"Pipeline Integration"**:
  <https://mlflow.org/docs/3.3.0/ml/traditional-ml/sklearn/guide>

La guía dice que los `Pipeline` de scikit-learn son "ciudadanos de primera clase" en MLflow. Con una
sola línea, `mlflow.sklearn.autolog()`, cada `fit` queda registrado con:

| Qué | Detalle |
|---|---|
| Parámetros | Todo lo que devuelve `get_params(deep=True)`: los de **cada paso** del pipeline |
| Métricas | Las de entrenamiento y las que se calculan después con `sklearn.metrics` o `score()` |
| Modelo | El pipeline completo, con su firma y un ejemplo de entrada |
| Búsquedas | Con `GridSearchCV` o `RandomizedSearchCV`, un run hijo por cada una de las mejores combinaciones (`max_tuning_runs`) |

### Análisis del código de ejemplo

El ejemplo de la guía arma un `ColumnTransformer` (escalado + `SelectKBest` para las numéricas y
`OneHotEncoder` para las categóricas), lo une a una `LinearRegression` en un `Pipeline`, activa
`autolog()` y entrena dentro de `mlflow.start_run()`. Lo que nos llamó la atención:

1. **No hace falta tocar el pipeline.** MLflow intercepta `fit`, así que el mismo código funciona con y
   sin seguimiento.
2. **Los nombres de los parámetros siguen la ruta del pipeline** (`preprocessor__num__scaler__...`),
   la misma sintaxis `paso__parametro` que usa `GridSearchCV`. Por eso en MLflow se puede filtrar por
   cualquier paso.
3. **El ejemplo no corre tal cual:** usa `X_train` y `y_train` sin definirlos, y supone columnas
   (`age`, `income`, `occupation`...) que no existen.
4. **`OneHotEncoder(drop="first")` sin `handle_unknown`** falla si en prueba aparece una categoría que
   no estaba en entrenamiento. En nuestro dataset pasa con los equipos, por eso nuestro pipeline usa
   `handle_unknown="ignore"`.
5. **`autolog` solo ve lo que pasa dentro de scikit-learn**: registra los datos como un dataset
   genérico, sin decir de qué archivo vienen, y sus métricas y gráficas son con los datos de
   entrenamiento. Lo demás hay que registrarlo a mano (sección 3).

En este notebook lo aplicamos a nuestro pipeline de las dos formas: automática y manual.

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import mlflow
import pandas as pd
from mlflow import MlflowClient
from sklearn import set_config
from sklearn.utils import estimator_html_repr

from taller3_pipeline import (ALIAS, EXPERIMENTO, FAMILIAS, MODELO_REGISTRADO, SEED, asignar_alias,
                              cargar_datos, configurar_mlflow, construir_pipeline, entrenar_familia,
                              espacio_del_pipeline, servidor_disponible)

URI = "http://127.0.0.1:8080"
print("MLflow", mlflow.__version__, "· ¿servidor en línea?", servidor_disponible(URI))
experimento = configurar_mlflow(EXPERIMENTO, URI)
client = MlflowClient()

2026/10/13 18:46:07 INFO mlflow.tracking.fluent: Experiment with name 'Champions_Resultado' does not exist. Creating a new experiment.


MLflow 3.17.1 · ¿servidor en línea? True


## 1. Los datos y el pipeline

Son los 144 partidos del Ejercicio 1, sin `score` ni `winner` (revelan el resultado) y divididos en
entrenamiento y prueba de forma estratificada. El pipeline convierte a número las columnas que vienen
como texto (`'63%'`, `'3 of 10'`), imputa, escala, codifica los equipos, selecciona variables y
termina en el clasificador.

In [2]:
X_train, X_test, y_train, y_test = cargar_datos()
print("Entrenamiento:", X_train.shape, "· prueba:", X_test.shape)
y_train.value_counts().to_frame("partidos (entrenamiento)")

Entrenamiento: (115, 12) · prueba: (29, 12)


,partidos (entrenamiento)
result,
Home Win,57
Away Win,38
Draw,20


In [3]:
set_config(display="diagram")
pipe = construir_pipeline()
Path("../pipeline_diagram.html").write_text(estimator_html_repr(pipe), encoding="utf-8")
print("Parámetros que expone get_params(deep=True):", len(pipe.get_params(deep=True)))
pipe

Parámetros que expone get_params(deep=True): 106


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('seleccion', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('porcentajes', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_thr

## 2. Registro automático con `mlflow.sklearn.autolog()`

Calibramos el bosque aleatorio con `RandomizedSearchCV` (20 combinaciones, validación cruzada de 5
particiones, F1 macro) con el autolog encendido. No hay ninguna llamada `log_*`: solo `fit`, y después
el F1 de prueba calculado con `sklearn.metrics`, que el autolog registra solo.

**Cuidado con el orden de los imports:** `autolog()` reemplaza las funciones de `sklearn.metrics` en el
momento en que se activa. Si antes se hizo `from sklearn.metrics import f1_score`, ese nombre sigue
apuntando a la función original y la métrica de prueba **no se registra** (nos pasó en la primera
corrida). Por eso aquí se llama como `metrics.f1_score`.

In [4]:
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

mlflow.sklearn.autolog(max_tuning_runs=5)

bosque = FAMILIAS["random_forest"]()
busqueda = RandomizedSearchCV(
    construir_pipeline(bosque["estimador"]),
    espacio_del_pipeline(bosque["espacio"]),
    n_iter=20,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=-1,
)

with mlflow.start_run(run_name="autolog-random-forest") as run_auto:
    busqueda.fit(X_train, y_train)
    f1_prueba = metrics.f1_score(y_test, busqueda.predict(X_test), average="macro")

mlflow.sklearn.autolog(disable=True)
print(f"Mejor F1 macro (validación cruzada): {busqueda.best_score_:.3f} · F1 macro en prueba: {f1_prueba:.3f}")

2026/10/13 18:46:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/13 18:46:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/13 18:46:53 INFO mlflow.sklearn.utils: Logging the 5 best runs, 15 runs will be omitted.


🏃 View run charming-boar-699 at: http://127.0.0.1:8080/#/experiments/3/runs/2594e55046bc42ea85884bdb7d428f1a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
🏃 View run agreeable-conch-408 at: http://127.0.0.1:8080/#/experiments/3/runs/b020e7d25b7d4e5e9df479e3591a559f
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
🏃 View run mysterious-hound-598 at: http://127.0.0.1:8080/#/experiments/3/runs/6f856b80d9a04cfe821b7fb466da581b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
🏃 View run mercurial-koi-954 at: http://127.0.0.1:8080/#/experiments/3/runs/c709acf26e1a45eea76224d90376d4c8
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
🏃 View run nimble-quail-473 at: http://127.0.0.1:8080/#/experiments/3/runs/862cfa7d009f469da57900c0aa7332e7
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


🏃 View run autolog-random-forest at: http://127.0.0.1:8080/#/experiments/3/runs/1d6a1306ff1a465f9fec7dd648bb58ec
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
Mejor F1 macro (validación cruzada): 0.576 · F1 macro en prueba: 0.536


Revisamos qué guardó MLflow en ese run:

In [5]:
run = client.get_run(run_auto.info.run_id)
hijos = mlflow.search_runs(experiment_ids=[experimento.experiment_id],
                           filter_string=f"tags.mlflow.parentRunId = '{run_auto.info.run_id}'")
modelos = client.search_logged_models(experiment_ids=[experimento.experiment_id],
                                      filter_string=f"source_run_id = '{run_auto.info.run_id}'")
print(f"RUN PADRE · {len(run.data.params)} parámetros:", ", ".join(sorted(run.data.params)))
print("Métricas:", ", ".join(f"{k} = {v:.3f}" for k, v in sorted(run.data.metrics.items())))
print("Archivos:", ", ".join(a.path for a in client.list_artifacts(run_auto.info.run_id)))
print("Modelos:", ", ".join(m.name for m in modelos))
print("Datasets:", [(d.dataset.name, d.dataset.source_type) for d in run.inputs.dataset_inputs])

hijo = client.get_run(hijos.iloc[0].run_id)
print()
print(f"RUNS HIJOS: {len(hijos)} · cada uno con {len(hijo.data.params)} parámetros")
print("Métricas de cada hijo:", ", ".join(sorted(hijo.data.metrics)))

RUN PADRE · 17 parámetros: best_modelo__class_weight, best_modelo__max_depth, best_modelo__max_features, best_modelo__min_samples_leaf, best_modelo__n_estimators, cv, error_score, estimator, n_iter, n_jobs, param_distributions, pre_dispatch, random_state, refit, return_train_score, scoring, verbose
Métricas: best_cv_score = 0.576, f1_score_X_test = 0.536, training_accuracy_score = 0.974, training_f1_score = 0.974, training_log_loss = 0.533, training_precision_score = 0.976, training_recall_score = 0.974, training_roc_auc = 0.999, training_score = 0.971
Archivos: cv_results.csv, estimator.html, metric_info.json, training_confusion_matrix.png
Modelos: best_estimator, model
Datasets: [('dataset', 'code'), ('dataset', 'code')]

RUNS HIJOS: 5 · cada uno con 106 parámetros
Métricas de cada hijo: mean_fit_time, mean_score_time, mean_test_score, rank_test_score, std_fit_time, std_score_time, std_test_score


In [6]:
# En los runs hijos sí está cada paso del pipeline, con su ruta completa
parametros = pd.Series(hijo.data.params)
parametros[parametros.index.str.contains(
    "numericas__imputar__strategy|onehot__handle_unknown|seleccion__k$|modelo__n_estimators|modelo__max_depth"
)].to_frame("valor")

,valor
preprocesamiento__numericas__imputar__strategy,median
preprocesamiento__categoricas__onehot__handle_unknown,ignore
seleccion__k,all
modelo__max_depth,10
modelo__n_estimators,359


**Lo que encontramos con el autolog** (capturas `06` y `08`):

- **La guía habla de `get_params(deep=True)`, pero con una búsqueda eso no queda en el run padre.** El
  padre tiene solo 17 parámetros: la configuración de `RandomizedSearchCV` y los mejores valores
  (`best_modelo__...`). Los 106 parámetros del pipeline, con su ruta completa, están en cada uno de los
  **5 runs hijos**, que son las 5 mejores de las 20 combinaciones (`max_tuning_runs=5`).
- **Las métricas del padre son de entrenamiento**, por eso se ven tan altas (`training_f1_score` =
  0.974). La de validación cruzada es `best_cv_score` = 0.576 y la de prueba, `f1_score_X_test` = 0.536.
  Sin calcular la de prueba, el run haría creer que el modelo es casi perfecto.
- **Guarda dos modelos**: la búsqueda completa (`model`) y el mejor pipeline (`best_estimator`), además
  de `cv_results.csv`, el diagrama `estimator.html` y una matriz de confusión de entrenamiento.
- **Los datasets aparecen como `dataset` con origen `code`**: sabe que hubo datos de entrenamiento y de
  evaluación, pero no de qué archivo salieron.
- **Guarda el modelo con pickle (cloudpickle)**, de ahí el aviso en la salida, al contrario de
  `log_model`, que usa skops. `autolog` no tiene el argumento `skops_trusted_types`, así que con
  nuestros transformadores propios no puede usar el formato seguro.

## 3. Registro manual: cada cosa a propósito

La función `entrenar_familia` (en `src/taller3_pipeline/seguimiento.py`) hace la misma búsqueda, pero
decide qué guardar:

- **Parámetros:** solo la configuración de la búsqueda y los mejores hiperparámetros, con nombres cortos.
- **Métricas:** F1 macro de validación cruzada (y su desviación), accuracy y F1 macro en prueba, y el
  F1 de un modelo que siempre dice "gana el local" como referencia.
- **Datasets** (`mlflow.log_input`): entrenamiento y prueba, con su huella digital y el CSV de origen.
- **Archivos:** matriz de confusión, reporte de clasificación y la tabla completa de la búsqueda.
- **Modelo:** el pipeline con su firma y con el código del paquete (`code_paths`), para que se pueda
  cargar en una máquina que no tenga instalado `taller3_pipeline`.

Corremos las cuatro familias de la Actividad 9 (40 combinaciones cada una) bajo un run padre:

In [7]:
resultados = []
with mlflow.start_run(run_name="comparacion-familias") as run_padre:
    for familia in FAMILIAS:
        resultados.append(entrenar_familia(familia, n_iter=40))

tabla = (pd.DataFrame([{"familia": r["familia"], **r["metricas"]} for r in resultados])
         .set_index("familia").sort_values("cv_f1_macro", ascending=False))
tabla.round(3)

🏃 View run random_forest at: http://127.0.0.1:8080/#/experiments/3/runs/443662a309bc4ea68b6a7017934503f8
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


🏃 View run regresion_logistica at: http://127.0.0.1:8080/#/experiments/3/runs/923b8a28da684fe3a895688ac2648ece
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


🏃 View run svc at: http://127.0.0.1:8080/#/experiments/3/runs/28e1b5fddaae448fbc1b1d63143ed8b9
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


🏃 View run hist_gradient_boosting at: http://127.0.0.1:8080/#/experiments/3/runs/e82197b8c88d49d3a8a4620638c09246
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
🏃 View run comparacion-familias at: http://127.0.0.1:8080/#/experiments/3/runs/16bd61d16a1c4ed690fee138c5b1d975
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


,cv_f1_macro,cv_f1_macro_std,test_accuracy,test_f1_macro,baseline_test_f1_macro
familia,,,,,
svc,0.638,0.052,0.655,0.601,0.217
regresion_logistica,0.611,0.073,0.621,0.542,0.217
hist_gradient_boosting,0.600,0.077,0.621,0.579,0.217
random_forest,0.583,0.048,0.621,0.576,0.217


In [8]:
# Lo que quedó en el run de la mejor familia
mejor = max(resultados, key=lambda r: r["metricas"]["cv_f1_macro"])
detalle = client.get_run(mejor["run_id"])
print("Familia:", mejor["familia"])
print("Parámetros:", detalle.data.params)
print("Datasets:", [(d.dataset.name, d.dataset.digest, [t.value for t in d.tags]) for d in detalle.inputs.dataset_inputs])
print("Archivos:", [a.path for a in client.list_artifacts(mejor["run_id"], "evaluacion")])

Familia: svc
Parámetros: {'familia': 'svc', 'n_iter': '40', 'cv_folds': '5', 'scoring': 'f1_macro', 'C': '0.17535949529764422', 'class_weight': 'balanced', 'gamma': '0.04609877941534893', 'kernel': 'linear'}
Datasets: [('champions-entrenamiento', 'efeb88e3', ['training']), ('champions-prueba', 'd3d05039', ['testing'])]
Archivos: ['evaluacion/matriz_confusion.png', 'evaluacion/reporte.txt']


**Resultados:** la **SVC** es la mejor por F1 macro de validación cruzada (0.638 ± 0.052) y también
en prueba (F1 macro 0.601, accuracy 0.655). Las cuatro familias superan con claridad a la referencia
que siempre dice "gana el local" (F1 macro 0.217). Son resultados muy parecidos a los de la Actividad 9
(allá la SVC tuvo 0.652 en validación y 0.604 en prueba, con 60 combinaciones por familia en lugar de
40).

Con el registro manual, el run de la SVC tiene 8 parámetros con nombres cortos, 5 métricas, los dos
datasets con su huella digital (`efeb88e3` el de entrenamiento) y la carpeta `evaluacion/` con la matriz
de confusión y el reporte (capturas `07`, `09` y `10`).

## 4. Automático contra manual

| | `autolog()` | Registro manual |
|---|---|---|
| Código | Una línea | Una llamada por cada cosa que se guarda |
| Parámetros | En el run padre, la configuración de la búsqueda y los mejores valores (17); en cada run hijo, los 106 del pipeline con la ruta completa | Solo los que importan, con nombres cortos (8 en el run de la SVC) |
| Métricas | Las de entrenamiento y las que se calculan con `sklearn.metrics` dentro del run | Las que definimos, incluida una referencia para comparar |
| Búsqueda | Un run hijo por cada una de las 5 mejores combinaciones | La tabla completa como CSV |
| Datos | Entrenamiento y evaluación como `dataset`, sin decir de qué archivo vienen | Entrenamiento y prueba con nombre, huella digital y archivo de origen |
| Gráficas | Matriz de confusión con los datos de entrenamiento | Matriz de confusión con los datos de prueba |

Nos quedamos con los dos: `autolog` para explorar rápido y el registro manual para los runs que se van a
comparar o a llevar a producción.

## 5. Registrar el mejor modelo (Model Registry)

El mejor pipeline por F1 de validación cruzada se registra como `champions-resultado`. Cada registro
crea una versión nueva; el alias `@campeon` apunta a la versión que se debe usar. Si mañana entrenamos
otra mejor, basta con mover el alias y quien cargue `models:/champions-resultado@campeon` recibe la
nueva sin cambiar su código.

In [9]:
version = mlflow.register_model(mejor["model_uri"], MODELO_REGISTRADO)
asignar_alias(version.version)
client.update_model_version(
    MODELO_REGISTRADO, version.version,
    description=f"{mejor['familia']} · F1 macro CV {mejor['metricas']['cv_f1_macro']:.3f} · "
                f"F1 macro prueba {mejor['metricas']['test_f1_macro']:.3f}",
)
client.set_model_version_tag(MODELO_REGISTRADO, version.version, "familia", mejor["familia"])
print(f"{MODELO_REGISTRADO} v{version.version} -> @{ALIAS}")

Successfully registered model 'champions-resultado'.


2026/10/13 18:48:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: champions-resultado, version 1


champions-resultado v1 -> @campeon


Created version '1' of model 'champions-resultado'.


Cargamos el modelo **por su alias**, como lo haría un servicio en producción, y comprobamos que
predice lo mismo que el pipeline del run:

In [10]:
from sklearn.metrics import f1_score

campeon = mlflow.sklearn.load_model(f"models:/{MODELO_REGISTRADO}@{ALIAS}")
prediccion = campeon.predict(X_test)
print(f"F1 macro en prueba del modelo cargado: {f1_score(y_test, prediccion, average='macro'):.3f} "
      f"(en el run: {mejor['metricas']['test_f1_macro']:.3f})")

firma = mlflow.models.get_model_info(f"models:/{MODELO_REGISTRADO}@{ALIAS}").signature
print("\nFirma de entrada:")
print(firma.inputs)

F1 macro en prueba del modelo cargado: 0.601 (en el run: 0.601)

Firma de entrada:
['home_shots_on_target_pct': double (required), 'away_shots_on_target_pct': double (required), 'home_saves_pct': double (required), 'away_saves_pct': double (required), 'home_possession': string (required), 'away_possession': string (required), 'home_shots_on_target': string (required), 'away_shots_on_target': string (required), 'home_saves': string (required), 'away_saves': string (required), 'home_team': string (required), 'away_team': string (required)]


In [11]:
# Un partido nuevo, con el mismo formato del CSV (texto incluido)
partido = X_test.head(3).copy()
pd.DataFrame({"local": partido["home_team"], "visitante": partido["away_team"],
              "predicción": campeon.predict(partido), "real": y_test.head(3).values})

,local,visitante,predicción,real
23,Marseille,Ajax,Home Win,Home Win
25,Galatasaray,Liverpool,Draw,Home Win
62,Olympiacos,PSV,Draw,Draw


## 6. El mismo entrenamiento como MLflow Project

En la raíz del repositorio está el archivo `MLproject`, que define dos *entry points* (`main` y
`registrar`) con sus parámetros y el comando que corren (`taller3-entrenar`). Con eso, cualquiera
puede reproducir el entrenamiento con `mlflow run`, sin saber qué script llamar:

```bash
mlflow run . --env-manager local -P familia=svc -P n_iter=20
```

`--env-manager local` usa el ambiente activo; sin esa opción MLflow crearía uno nuevo a partir de
`environment.yml`. Lo corremos desde aquí:

In [12]:
import subprocess

# PATH: para encontrar taller3-entrenar · PYTHONIOENCODING: MLflow imprime emojis y la consola de
# Windows (cp1252) no los acepta
scripts = str(Path(sys.executable).parent / "Scripts")
# Se quitan las variables del run activo del notebook: mlflow run abre su propio run
entorno = {k: v for k, v in os.environ.items() if k not in ("MLFLOW_EXPERIMENT_ID", "MLFLOW_RUN_ID")}
entorno.update({"MLFLOW_TRACKING_URI": URI, "PATH": scripts + os.pathsep + os.environ["PATH"],
                "MLFLOW_DISABLE_AGENT_HINT": "1", "PYTHONIOENCODING": "utf-8"})
salida = subprocess.run(
    [sys.executable, "-m", "mlflow", "run", str(Path("..").resolve()), "--env-manager", "local",
     "--experiment-name", EXPERIMENTO, "-P", "familia=svc", "-P", "n_iter=20"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", env=entorno,
)
if salida.returncode != 0:
    print(salida.stderr[-2000:])
print(salida.stdout.strip())
print("\n".join(linea for linea in salida.stderr.splitlines() if "mlflow.projects" in linea))

MLflow: http://127.0.0.1:8080 · experimento Champions_Resultado
  calibrando svc (20 combinaciones)...
🏃 View run svc at: http://127.0.0.1:8080/#/experiments/3/runs/3ad88725a01d450b857eb9333f40255d
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3

familia                  CV F1 macro  test acc   test F1  run_id
svc                            0.612     0.655     0.564  3ad88725a01d450b857eb9333f40255d
🏃 View run svc at: http://127.0.0.1:8080/#/experiments/3/runs/3ad88725a01d450b857eb9333f40255d
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3
2026/10/13 18:48:11 INFO mlflow.projects.utils: === Created directory C:\Users\dlinares\AppData\Local\Temp\tmpf3q890wq for downloading remote URIs passed to arguments of type 'path' ===
2026/10/13 18:48:11 INFO mlflow.projects.backend.local: === Running command 'taller3-entrenar --familia svc --n-iter 20' in run with ID '3ad88725a01d450b857eb9333f40255d' === 
2026/10/13 18:48:55 INFO mlflow.projects: === Run (ID '3ad88725a01d4

In [13]:
print("Experimento en la interfaz:", f"{URI}/#/experiments/{experimento.experiment_id}")
print("Modelo registrado:", f"{URI}/#/models/{MODELO_REGISTRADO}")

Experimento en la interfaz: http://127.0.0.1:8080/#/experiments/3
Modelo registrado: http://127.0.0.1:8080/#/models/champions-resultado


## Conclusiones

- Conectar MLflow con un pipeline de scikit-learn no exige cambiar el pipeline: el autolog intercepta
  `fit`, y el registro manual solo envuelve el entrenamiento en `mlflow.start_run`.
- El autolog sirve para explorar, pero hay que saber leerlo: con una búsqueda, el detalle del pipeline
  está en los runs hijos y las métricas del run padre son de entrenamiento.
- Para comparar modelos en serio conviene el registro manual: decidimos qué métrica importa (F1 macro),
  guardamos una referencia contra la cual comparar, los datos exactos y las gráficas de prueba.
- El Model Registry separa el modelo que se entrenó del modelo que se usa: el código que carga
  `models:/champions-resultado@campeon` no cambia cuando mejoramos el modelo (captura `11`).
- Con el `MLproject`, el mismo entrenamiento se reproduce con un comando (`mlflow run .`), y MLflow
  marca ese run con origen `PROJECT` (el primer run `svc` de la captura `06`).
- En MLflow 3.17 los modelos de `log_model` se guardan con skops y hay que declarar los tipos
  confiables; el autolog sigue usando pickle.